# prime-rl S6: LFM2.5-VL pipeline check (RTX 6000 Pro offline)

Each test cell prints PASS/FAIL.

| Test | Goal |
|---|---|
| A | Introspect Lfm2VlForConditionalGeneration |
| B | prime-rl SFT trainer can load LFM2.5-VL |
| C | prime-rl SFT trainer runs 2 steps on synthetic VLM data |
| D | prime-rl inference (vLLM) serves LFM2.5-VL |

## 1. Install (S2 recipe) + patches

In [ ]:
import os, subprocess, glob, re

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
WHEELS = f"{BASE}/wheels"
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"

SKIP_PREFIXES = (
    "torch-", "torchvision-", "torchaudio-", "nvidia_",
    "numpy-", "scipy-", "scikit_learn-", "pandas-",
)

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m: return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]
keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w); continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)
filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")
subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)

patch_registry = '''
import inspect
import prime_rl.utils.vlm as v
path = inspect.getfile(v)
content = open(path).read()
if "lfm2_vl" not in content:
    needle = "    \\"qwen3_vl\\":"
    insertion = (
        "    \\"lfm2_vl\\": VLMModelInfo(vision_encoder_attr=\\"model.vision_tower\\", language_model_attr=\\"model.language_model\\"),\\n"
        "    \\"qwen3_vl\\":"
    )
    open(path, "w").write(content.replace(needle, insertion, 1))
    print(f"PATCH 1 ok: {path}")
else:
    print("PATCH 1 skip")
'''
subprocess.run(["python3.12", "-c", patch_registry], check=True)

patch_moe_guard = '''
import inspect
import prime_rl.trainer.model as m
path = inspect.getfile(m)
content = open(path).read()
old_block = "    for transformer_block in language_model.layers:\\n        if not isinstance(transformer_block.mlp, (MoE, LatentMoE)):\\n            continue\\n        transformer_block.mlp.set_ep_comm_backend(backend)\\n        transformer_block.mlp.set_deepep_token_chunk_size(config.deepep_token_chunk_size)"
new_block = "    for transformer_block in language_model.layers:\\n        mlp = getattr(transformer_block, \\"mlp\\", None)\\n        if mlp is None or not isinstance(mlp, (MoE, LatentMoE)):\\n            continue\\n        mlp.set_ep_comm_backend(backend)\\n        mlp.set_deepep_token_chunk_size(config.deepep_token_chunk_size)"
if old_block in content:
    open(path, "w").write(content.replace(old_block, new_block))
    print(f"PATCH 2 ok: {path}")
elif "mlp = getattr(transformer_block" in content:
    print("PATCH 2 skip (already)")
else:
    print("PATCH 2 WARN: needle not found")
'''
subprocess.run(["python3.12", "-c", patch_moe_guard], check=True)

subprocess.run(
    "python3.12 -c 'from prime_rl.utils.vlm import VLM_REGISTRY; print(\"VLM_REGISTRY:\", list(VLM_REGISTRY.keys()))'",
    shell=True, check=True,
)

## Test A: introspect

In [ ]:
import os, torch
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"

from transformers import AutoConfig, AutoModelForImageTextToText
cfg = AutoConfig.from_pretrained(MODEL_DIR)
with torch.device("meta"):
    model = AutoModelForImageTextToText.from_config(cfg)
print(f"model class: {type(model).__name__}")
for name, child in model.named_children():
    print(f"  model.{name}  ({type(child).__name__})")
if hasattr(model, "model"):
    for name, child in model.model.named_children():
        print(f"  model.model.{name}  ({type(child).__name__})")
print("\nA: PASS")

## Test B: prime-rl SFT loads LFM2.5-VL (no train)

Uses the synthetic VLM dataset built below as data source (since `[data.fake]`
doesn't apply to SFT trainer).

In [ ]:
import os, subprocess
from PIL import Image
from datasets import Dataset

DATA_DIR = "/kaggle/working/synthetic_vlm"
os.makedirs(DATA_DIR, exist_ok=True)
rows = []
color_map = {(255,0,0):"red",(0,255,0):"green",(0,0,255):"blue",(200,200,0):"yellow"}
for i, color in enumerate(color_map.keys()):
    img = Image.new("RGB", (128, 128), color=color)
    name = f"img_{i}.png"
    img.save(f"{DATA_DIR}/{name}")
    rows.append({
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "path": f"{DATA_DIR}/{name}"},
                {"type": "text", "text": "What color?"},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": color_map[color]},
            ]},
        ],
    })
ds = Dataset.from_list(rows)

# Save as parquet under a folder with a `train/` split-style subdir so
# `load_dataset(folder_path)` auto-detects via the parquet builder.
# `save_to_disk` would NOT work because prime-rl uses `load_dataset`, not
# `load_from_disk`.
PARQUET_DIR = f"{DATA_DIR}/parquet"
os.makedirs(f"{PARQUET_DIR}/train", exist_ok=True)
ds.to_parquet(f"{PARQUET_DIR}/train/data.parquet")
subprocess.run(f"find {PARQUET_DIR} -type f", shell=True)
print(f"\nrows = {len(ds)}, columns = {ds.column_names}")

In [ ]:
import os, subprocess, time

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"
VISION_ATTR = "model.vision_tower"
LM_ATTR     = "model.language_model"

# ckpt section omitted to disable checkpoint saving (don't bloat outputs).
TOML_B = f'''max_steps = 0

[model]
name = "{MODEL_DIR}"
seq_len = 256
attn = "sdpa"
optimization_dtype = "bfloat16"
reduce_dtype = "bfloat16"

[model.vlm]
vision_encoder_attr = "{VISION_ATTR}"
language_model_attr = "{LM_ATTR}"

[data]
name = "/kaggle/working/synthetic_vlm/parquet"
seq_len = 256
batch_size = 1

[optim]
lr = 1e-6
'''
os.makedirs("/kaggle/working/outputs", exist_ok=True)
toml_b_path = "/kaggle/working/outputs/sft_b.toml"
log_b       = "/kaggle/working/outputs/sft_b.log"
with open(toml_b_path, "w") as f:
    f.write(TOML_B)
print("=== sft_b.toml ===")
print(TOML_B)

env = os.environ.copy()
env.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1", "WANDB_MODE": "disabled"})

t0 = time.time()
with open(log_b, "wb") as lf:
    result = subprocess.run(
        f"python3.12 -m prime_rl.entrypoints.sft @ {toml_b_path}",
        shell=True, env=env, stdout=lf, stderr=subprocess.STDOUT,
    )
elapsed = time.time() - t0
print(f"\nB: exit={result.returncode}  elapsed={elapsed:.1f}s")
print("--- sft_b.log tail ---")
subprocess.run(f"tail -n 80 {log_b}", shell=True)
print("B: " + ("PASS" if result.returncode == 0 else "FAIL"))

## Test C: SFT 2 steps on synthetic VLM data

In [ ]:
import os, subprocess, time

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"
VISION_ATTR = "model.vision_tower"
LM_ATTR     = "model.language_model"

# No [ckpt] section — let prime-rl decide. If a default still saves, the
# manifest cell at the end will reveal it and we can switch off explicitly.
TOML_C = f'''max_steps = 2

[model]
name = "{MODEL_DIR}"
seq_len = 512
attn = "sdpa"
optimization_dtype = "bfloat16"
reduce_dtype = "bfloat16"

[model.vlm]
vision_encoder_attr = "{VISION_ATTR}"
language_model_attr = "{LM_ATTR}"

[data]
name = "/kaggle/working/synthetic_vlm/parquet"
seq_len = 512
batch_size = 1

[optim]
lr = 2e-5
'''
os.makedirs("/kaggle/working/outputs", exist_ok=True)
toml_c_path = "/kaggle/working/outputs/sft_c.toml"
log_c       = "/kaggle/working/outputs/sft_c.log"
with open(toml_c_path, "w") as f:
    f.write(TOML_C)
print("=== sft_c.toml ===")
print(TOML_C)

env = os.environ.copy()
env.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1", "WANDB_MODE": "disabled"})

t0 = time.time()
with open(log_c, "wb") as lf:
    result = subprocess.run(
        f"python3.12 -m prime_rl.entrypoints.sft @ {toml_c_path}",
        shell=True, env=env, stdout=lf, stderr=subprocess.STDOUT,
    )
elapsed = time.time() - t0
print(f"\nC: exit={result.returncode}  elapsed={elapsed:.1f}s")
print("--- sft_c.log tail ---")
subprocess.run(f"tail -n 120 {log_c}", shell=True)
print("C: " + ("PASS" if result.returncode == 0 else "FAIL"))

## Test D: vLLM serves LFM2.5-VL

In [ ]:
import os, subprocess, time, urllib.request

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/LFM2.5-VL-450M"

INFER_TOML = f'''gpu_memory_utilization = 0.5

[model]
name = "{MODEL_DIR}"
max_model_len = 512
enforce_eager = true

[server]
port = 8000
'''
os.makedirs("/kaggle/working/outputs", exist_ok=True)
infer_toml_path = "/kaggle/working/outputs/infer_d.toml"
log_path        = "/kaggle/working/outputs/inference_d.log"
with open(infer_toml_path, "w") as f:
    f.write(INFER_TOML)
print("=== infer_d.toml ===")
print(INFER_TOML)

env = os.environ.copy()
env.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "WANDB_MODE": "disabled", "VLLM_API_KEY": "dummy"})

lf = open(log_path, "wb")
p = subprocess.Popen(
    f"python3.12 -m prime_rl.entrypoints.inference @ {infer_toml_path}",
    shell=True, env=env, stdout=lf, stderr=subprocess.STDOUT,
)
ready = False
deadline = time.time() + 600
try:
    while time.time() < deadline:
        if p.poll() is not None: break
        try:
            req = urllib.request.Request("http://localhost:8000/v1/models", headers={"Authorization": "Bearer dummy"})
            if urllib.request.urlopen(req, timeout=5).status == 200:
                ready = True; break
        except Exception: pass
        time.sleep(5)
    print(f"\nD: vLLM ready = {ready}")
    if not ready:
        print("--- inference log tail ---")
        subprocess.run(f"tail -n 80 {log_path}", shell=True)
    print("D: " + ("PASS" if ready else "FAIL"))
finally:
    if p.poll() is None:
        p.terminate(); p.wait(timeout=10)
    lf.close()

## Manifest: what actually survived in /kaggle/working/

If a file is in `/kaggle/working/outputs/` it should appear in
`kaggle kernels output`. Anything missing tells us what kaggle drops.

In [ ]:
import subprocess, shutil, os
os.makedirs("/kaggle/working/outputs", exist_ok=True)
manifest = "/kaggle/working/outputs/manifest.txt"
with open(manifest, "w") as f:
    f.write("# files under /kaggle/working/ at end of notebook\n")
    for root, dirs, files in os.walk("/kaggle/working"):
        for fn in files:
            p = os.path.join(root, fn)
            try: sz = os.path.getsize(p)
            except OSError: sz = -1
            f.write(f"{sz:>12d}  {p}\n")
print("=== manifest tail ===")
subprocess.run(f"tail -n 80 {manifest}", shell=True)

# Also hard-delete any checkpoint dir to keep output small in case prime-rl
# wrote one despite the omitted [ckpt] block.
ckpt_dir = "/kaggle/working/outputs/checkpoints"
if os.path.isdir(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"removed: {ckpt_dir}")
else:
    print(f"no ckpt dir: {ckpt_dir}")